# Notebook B — Baseline Comparison Grid (7 models × 3 features)

**Purpose:** systematic comparison of 7 architectures × 3 feature
representations on the OAHS 5-class VHD task. All 21 experiments share
identical training settings and fold splits.

## Models
| # | Name | Architecture |
|---|---|---|
| 1 | CNN-1 | 1× Conv2D block → GAP → Dense |
| 2 | CNN-2 | 2× Conv2D blocks → GAP → Dense |
| 3 | SepCNN-1 | 1× SeparableConv2D block → GAP → Dense |
| 4 | SepCNN-2 | 2× SeparableConv2D blocks → GAP → Dense |
| 5 | Conv-LSTM | 2× Conv blocks → Reshape → LSTM(32) → Dense |
| 6 | Conv-BiLSTM | 2× Conv blocks → Reshape → BiLSTM(32) → Dense |
| 7 | Conv-GRU | 2× Conv blocks → Reshape → GRU(32) → Dense |

## Features (loaded from Notebook A cache)
- DWT scalogram (6, 94, 1)
- MFCC (40, 94, 1)
- Log-mel spectrogram (64, 94, 1)

## Outputs
- `results_master.csv` — one row per (model, feature) pair with all metrics
- `confusion_matrices.npz` — per-experiment aggregate confusion matrix
- `histories.json` — per-experiment training histories
- Plots: params-vs-acc, FLOPs-vs-acc, results heatmap

## Resume safety
The notebook checkpoints `results_master.csv` after **every** experiment.
If Colab disconnects, re-run from the top — completed experiments are skipped.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
import os, time, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Configuration

All training settings shared across every experiment. Only architecture
and feature vary.

In [ ]:
# ---------- Paths ----------
FEAT_DIR   = '/content/drive/MyDrive/Msc_ML_project/features_v1'
OUTPUT_DIR = '/content/drive/MyDrive/Msc_ML_project/results_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Writable:', os.access(OUTPUT_DIR, os.W_OK))

# ---------- Classes ----------
CLASSES = ['AS', 'MR', 'MS', 'MVP', 'N']
NUM_CLASSES = len(CLASSES)

# ---------- Architecture defaults ----------
N_FILTERS    = 32      # filters per conv layer
KERNEL       = (3, 3)
POOL         = (2, 2)
RNN_UNITS    = 32      # for LSTM/BiLSTM/GRU
DROPOUT      = 0.3

# ---------- Training ----------
N_FOLDS         = 5
EPOCHS          = 100
BATCH_SIZE      = 32
LR_BASE         = 3e-4
LR_WARMUP_INIT  = 5e-5
WARMUP_EPOCHS   = 5
ES_PATIENCE     = 20
ES_MIN_EPOCHS   = 25
RLR_PATIENCE    = 10

EPS = 1e-10

print('Config loaded.')
print(f'  EPOCHS={EPOCHS}  BATCH={BATCH_SIZE}  LR={LR_BASE}')
print(f'  N_FOLDS={N_FOLDS}  ES_PATIENCE={ES_PATIENCE}')

## 3. Load cached features and folds from Notebook A

In [ ]:
def load_feature(name):
    p = os.path.join(FEAT_DIR, f'oahs_{name}.npz')
    d = np.load(p)
    return d['X'], d['y']

X_dwt,  y = load_feature('dwt')
X_mfcc, _ = load_feature('mfcc')
X_spec, _ = load_feature('logmel')

folds = np.load(os.path.join(FEAT_DIR, 'oahs_folds.npz'), allow_pickle=True)
fold_train = [np.asarray(a, dtype=np.int64) for a in folds['train_idx']]
fold_val   = [np.asarray(a, dtype=np.int64) for a in folds['val_idx']]

# Map a feature name to its (X, input_shape) for the registry
FEATURE_REGISTRY = {
    'DWT':    (X_dwt,  X_dwt.shape[1:]),
    'MFCC':   (X_mfcc, X_mfcc.shape[1:]),
    'LogMel': (X_spec, X_spec.shape[1:]),
}

print('Features loaded:')
for k, (X, s) in FEATURE_REGISTRY.items():
    print(f'  {k:>7}: X.shape={X.shape}  input_shape={s}')
print(f'  y: {y.shape}  classes: {np.bincount(y).tolist()}')
print(f'  Folds: {len(fold_train)}  (sizes: train={[len(t) for t in fold_train]}, val={[len(v) for v in fold_val]})')

## 4. Model factory — 7 architectures

Each builder takes `input_shape` and `num_classes` and returns a compiled
`tf.keras.Model`. Same conv-block style across all conv-based models.
All use the same dense head (GAP → Dropout → Dense Softmax) for fair
comparison — except recurrent models which use the recurrent output.

In [ ]:
def _conv_block(x, filters, sep=False, prefix='b'):
    """Conv2D or SeparableConv2D + BN + ReLU + MaxPool."""
    if sep:
        x = layers.SeparableConv2D(filters, KERNEL, padding='same',
                                   use_bias=False, name=f'{prefix}_sepconv')(x)
    else:
        x = layers.Conv2D(filters, KERNEL, padding='same',
                          use_bias=False, name=f'{prefix}_conv')(x)
    x = layers.BatchNormalization(name=f'{prefix}_bn')(x)
    x = layers.ReLU(name=f'{prefix}_relu')(x)
    x = layers.MaxPooling2D(POOL, padding='same', name=f'{prefix}_pool')(x)
    return x


def build_cnn_1(input_shape, num_classes=NUM_CLASSES):
    inp = Input(shape=input_shape, name='input')
    x = _conv_block(inp, N_FILTERS, sep=False, prefix='b1')
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='probs')(x)
    return Model(inp, out, name='CNN-1')


def build_cnn_2(input_shape, num_classes=NUM_CLASSES):
    inp = Input(shape=input_shape, name='input')
    x = _conv_block(inp, N_FILTERS, sep=False, prefix='b1')
    x = _conv_block(x,   N_FILTERS, sep=False, prefix='b2')
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='probs')(x)
    return Model(inp, out, name='CNN-2')


def build_sepcnn_1(input_shape, num_classes=NUM_CLASSES):
    inp = Input(shape=input_shape, name='input')
    x = _conv_block(inp, N_FILTERS, sep=True, prefix='b1')
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='probs')(x)
    return Model(inp, out, name='SepCNN-1')


def build_sepcnn_2(input_shape, num_classes=NUM_CLASSES):
    inp = Input(shape=input_shape, name='input')
    x = _conv_block(inp, N_FILTERS, sep=True, prefix='b1')
    x = _conv_block(x,   N_FILTERS, sep=True, prefix='b2')
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='probs')(x)
    return Model(inp, out, name='SepCNN-2')


def _conv_to_seq(x):
    """Reshape (H', W', C) → (W', H'*C) so the recurrent layer sees
    `W'` time steps with `H'*C` features each (time = horizontal axis)."""
    h, w, c = x.shape[1], x.shape[2], x.shape[3]
    x = layers.Permute((2, 1, 3), name='to_time_first')(x)   # (W', H', C)
    x = layers.Reshape((w, h * c), name='to_seq')(x)         # (W', H'*C)
    return x


def build_conv_lstm(input_shape, num_classes=NUM_CLASSES):
    inp = Input(shape=input_shape, name='input')
    x = _conv_block(inp, N_FILTERS, sep=False, prefix='b1')
    x = _conv_block(x,   N_FILTERS, sep=False, prefix='b2')
    x = _conv_to_seq(x)
    x = layers.LSTM(RNN_UNITS, name='lstm')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='probs')(x)
    return Model(inp, out, name='Conv-LSTM')


def build_conv_bilstm(input_shape, num_classes=NUM_CLASSES):
    inp = Input(shape=input_shape, name='input')
    x = _conv_block(inp, N_FILTERS, sep=False, prefix='b1')
    x = _conv_block(x,   N_FILTERS, sep=False, prefix='b2')
    x = _conv_to_seq(x)
    x = layers.Bidirectional(layers.LSTM(RNN_UNITS), name='bilstm')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='probs')(x)
    return Model(inp, out, name='Conv-BiLSTM')


def build_conv_gru(input_shape, num_classes=NUM_CLASSES):
    inp = Input(shape=input_shape, name='input')
    x = _conv_block(inp, N_FILTERS, sep=False, prefix='b1')
    x = _conv_block(x,   N_FILTERS, sep=False, prefix='b2')
    x = _conv_to_seq(x)
    x = layers.GRU(RNN_UNITS, name='gru')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='probs')(x)
    return Model(inp, out, name='Conv-GRU')


MODEL_REGISTRY = {
    'CNN-1':       build_cnn_1,
    'CNN-2':       build_cnn_2,
    'SepCNN-1':    build_sepcnn_1,
    'SepCNN-2':    build_sepcnn_2,
    'Conv-LSTM':   build_conv_lstm,
    'Conv-BiLSTM': build_conv_bilstm,
    'Conv-GRU':    build_conv_gru,
}
print('Model registry:', list(MODEL_REGISTRY.keys()))

### Quick sanity check — build each model on each feature shape

In [ ]:
for model_name, builder in MODEL_REGISTRY.items():
    for feat_name, (_, shape) in FEATURE_REGISTRY.items():
        m = builder(shape)
        params = int(sum(np.prod(v.shape) for v in m.trainable_weights))
        print(f'  {model_name:>11} on {feat_name:>7}: input={shape}, params={params:,}')
    print()

## 5. FLOPs estimator

Analytical estimate (forward pass). Counts multiply-accumulate ops × 2.

In [ ]:
def estimate_flops(model):
    """Estimate forward-pass FLOPs by walking the model's layers."""
    total = 0
    # We need the output shape of each layer to know spatial dims
    for layer in model.layers:
        cfg = layer.get_config()
        out_shape = layer.output_shape
        in_shape  = layer.input_shape if not isinstance(layer.input_shape, list) else layer.input_shape[0]

        if isinstance(layer, layers.Conv2D) and not isinstance(layer, layers.SeparableConv2D):
            # Standard Conv2D: H_out * W_out * C_out * (kH * kW * C_in)
            H_out, W_out, C_out = out_shape[1], out_shape[2], out_shape[3]
            kH, kW = layer.kernel_size
            C_in   = in_shape[3]
            total += 2 * H_out * W_out * C_out * kH * kW * C_in
        elif isinstance(layer, layers.SeparableConv2D):
            H_out, W_out, C_out = out_shape[1], out_shape[2], out_shape[3]
            kH, kW = layer.kernel_size
            C_in   = in_shape[3]
            # depthwise
            total += 2 * H_out * W_out * C_in * kH * kW
            # pointwise
            total += 2 * H_out * W_out * C_out * C_in
        elif isinstance(layer, layers.Dense):
            n_in  = in_shape[-1]
            n_out = out_shape[-1]
            total += 2 * n_in * n_out
        elif isinstance(layer, layers.LSTM):
            # LSTM: 4 gates × (input + recurrent) matmuls × time steps
            T = in_shape[1]
            n_in = in_shape[2]
            units = layer.units
            # per step: 4 * (n_in + units) * units MACs
            total += 2 * T * 4 * (n_in + units) * units
        elif isinstance(layer, layers.Bidirectional):
            inner = layer.layer
            T = in_shape[1]
            n_in = in_shape[2]
            units = inner.units
            # Bidirectional = 2× the LSTM cost
            total += 2 * 2 * T * 4 * (n_in + units) * units
        elif isinstance(layer, layers.GRU):
            T = in_shape[1]
            n_in = in_shape[2]
            units = layer.units
            # GRU has 3 gates per step
            total += 2 * T * 3 * (n_in + units) * units
        elif isinstance(layer, layers.BatchNormalization):
            # ~2 ops per element
            shape = out_shape[1:]
            total += 2 * int(np.prod([s for s in shape if s is not None]))
    return int(total)


# Quick test
for mname, builder in MODEL_REGISTRY.items():
    for fname, (_, shape) in FEATURE_REGISTRY.items():
        m = builder(shape)
        f = estimate_flops(m)
        print(f'  {mname:>11} on {fname:>7}: ~{f/1e6:.2f} MFLOPs')
    print()

## 6. Training utilities — shared by all experiments

In [ ]:
def warmup_schedule(epoch, lr):
    if epoch < WARMUP_EPOCHS:
        frac = (epoch + 1) / WARMUP_EPOCHS
        return float(LR_WARMUP_INIT + (LR_BASE - LR_WARMUP_INIT) * frac)
    return float(lr)


class MinEpochsEarlyStopping(EarlyStopping):
    def __init__(self, min_epochs=0, **kwargs):
        super().__init__(**kwargs)
        self.min_epochs = min_epochs

    def on_epoch_end(self, epoch, logs=None):
        if epoch < self.min_epochs:
            self.wait = 0
            return
        super().on_epoch_end(epoch, logs)


def train_one_fold(builder, input_shape, X, y, tr_idx, val_idx, fold_seed):
    """Train one model on one fold. Returns (model, metrics, cm, history)."""
    tf.keras.backend.clear_session()
    tf.random.set_seed(fold_seed)
    np.random.seed(fold_seed)
    random.seed(fold_seed)

    model = builder(input_shape)
    model.compile(
        optimizer=Adam(learning_rate=LR_WARMUP_INIT),
        loss='categorical_crossentropy',
        metrics=['accuracy'])

    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    y_tr_oh  = to_categorical(y_tr,  NUM_CLASSES).astype(np.float32)
    y_val_oh = to_categorical(y_val, NUM_CLASSES).astype(np.float32)

    callbacks = [
        LearningRateScheduler(warmup_schedule, verbose=0),
        MinEpochsEarlyStopping(
            min_epochs=ES_MIN_EPOCHS,
            monitor='val_accuracy', patience=ES_PATIENCE,
            restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=RLR_PATIENCE, min_lr=1e-6),
    ]

    t0 = time.time()
    hist = model.fit(
        X_tr, y_tr_oh,
        validation_data=(X_val, y_val_oh),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=0)
    train_time = time.time() - t0

    y_pred = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0).argmax(axis=1)
    metrics = {
        'acc':  accuracy_score(y_val, y_pred),
        'prec': precision_score(y_val, y_pred, average='macro', zero_division=0),
        'rec':  recall_score(y_val, y_pred, average='macro', zero_division=0),
        'f1':   f1_score(y_val, y_pred, average='macro', zero_division=0),
        'epochs_trained': len(hist.history['loss']),
        'train_time_s': train_time,
    }
    cm = confusion_matrix(y_val, y_pred, labels=list(range(NUM_CLASSES)))
    return model, metrics, cm, hist.history


def run_experiment(model_name, feature_name):
    """Run 5-fold CV for one (model, feature) pair."""
    builder = MODEL_REGISTRY[model_name]
    X, input_shape = FEATURE_REGISTRY[feature_name]

    # Build once to count params + FLOPs
    probe = builder(input_shape)
    n_params = int(sum(np.prod(v.shape) for v in probe.trainable_weights))
    flops    = estimate_flops(probe)
    del probe; tf.keras.backend.clear_session()

    per_fold = []
    cm_total = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    histories = []
    for fold_i, (tr, va) in enumerate(zip(fold_train, fold_val), start=1):
        seed = SEED + 1000 * (hash(model_name + feature_name) % 100) + fold_i
        _, met, cm, hist = train_one_fold(
            builder, input_shape, X, y, tr, va, seed)
        per_fold.append(met)
        cm_total += cm
        histories.append(hist)
        print(f'    fold {fold_i}: acc={met["acc"]:.4f}  '
              f'f1={met["f1"]:.4f}  ep={met["epochs_trained"]}  '
              f't={met["train_time_s"]:.1f}s')

    acc_arr  = np.array([m['acc']  for m in per_fold])
    prec_arr = np.array([m['prec'] for m in per_fold])
    rec_arr  = np.array([m['rec']  for m in per_fold])
    f1_arr   = np.array([m['f1']   for m in per_fold])
    times    = np.array([m['train_time_s'] for m in per_fold])

    row = {
        'model':           model_name,
        'feature':         feature_name,
        'params':          n_params,
        'flops':           flops,
        'acc_mean':        float(acc_arr.mean()),
        'acc_std':         float(acc_arr.std()),
        'prec_mean':       float(prec_arr.mean()),
        'prec_std':        float(prec_arr.std()),
        'rec_mean':        float(rec_arr.mean()),
        'rec_std':         float(rec_arr.std()),
        'f1_mean':         float(f1_arr.mean()),
        'f1_std':          float(f1_arr.std()),
        'mean_time_s':     float(times.mean()),
        'mean_epochs':     float(np.mean([m['epochs_trained'] for m in per_fold])),
    }
    return row, cm_total, histories

print('Training utilities defined.')

## 7. Run the 21-experiment grid with resume safety

After each `(model, feature)` pair completes, `results_master.csv` is
overwritten and `confusion_matrices.npz` is updated. If Colab disconnects,
just re-run this cell — completed experiments are skipped.

In [ ]:
RESULTS_CSV = os.path.join(OUTPUT_DIR, 'results_master.csv')
CMS_NPZ     = os.path.join(OUTPUT_DIR, 'confusion_matrices.npz')
HIST_JSON   = os.path.join(OUTPUT_DIR, 'histories.json')

# Load prior progress if present
if os.path.exists(RESULTS_CSV):
    results_df = pd.read_csv(RESULTS_CSV)
    done_keys = set(zip(results_df['model'], results_df['feature']))
    print(f'Resuming: {len(done_keys)} experiments already complete.')
else:
    results_df = pd.DataFrame()
    done_keys = set()
    print('Starting fresh.')

# Load prior CMs and histories if present
if os.path.exists(CMS_NPZ):
    prior_cms = dict(np.load(CMS_NPZ))
else:
    prior_cms = {}
all_cms = prior_cms.copy()

if os.path.exists(HIST_JSON):
    with open(HIST_JSON) as f:
        all_histories = json.load(f)
else:
    all_histories = {}

# Define the grid (order chosen to do the fast models first)
GRID_MODELS   = ['CNN-1', 'SepCNN-1', 'CNN-2', 'SepCNN-2',
                 'Conv-GRU', 'Conv-LSTM', 'Conv-BiLSTM']
GRID_FEATURES = ['DWT', 'MFCC', 'LogMel']
TOTAL = len(GRID_MODELS) * len(GRID_FEATURES)

i = 0
t_grid_start = time.time()
for model_name in GRID_MODELS:
    for feature_name in GRID_FEATURES:
        i += 1
        if (model_name, feature_name) in done_keys:
            print(f'[{i:2d}/{TOTAL}] {model_name:>11} × {feature_name:>7}  '
                  f'-- skipped (already done)')
            continue

        print(f'\n[{i:2d}/{TOTAL}] {model_name:>11} × {feature_name:>7}')
        t0 = time.time()
        row, cm, hists = run_experiment(model_name, feature_name)
        elapsed = time.time() - t0
        print(f'    DONE in {elapsed:.1f}s  '
              f'acc={row["acc_mean"]:.4f}±{row["acc_std"]:.4f}  '
              f'params={row["params"]:,}  flops={row["flops"]/1e6:.2f}M')

        # Append and checkpoint
        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
        results_df.to_csv(RESULTS_CSV, index=False)

        cm_key = f'{model_name}__{feature_name}'
        all_cms[cm_key] = cm
        np.savez_compressed(CMS_NPZ, **all_cms)

        all_histories[cm_key] = hists
        with open(HIST_JSON, 'w') as f:
            json.dump(all_histories, f)

        done_keys.add((model_name, feature_name))

grid_time = time.time() - t_grid_start
print(f'\nGrid complete in {grid_time/60:.1f} minutes.')
print(f'Results saved to {RESULTS_CSV}')

## 8. Results summary

In [ ]:
results_df = pd.read_csv(RESULTS_CSV)
display_cols = ['model', 'feature', 'params', 'flops',
                'acc_mean', 'acc_std', 'prec_mean', 'rec_mean', 'f1_mean',
                'mean_time_s']
print(results_df[display_cols].to_string(index=False))

print('\nBest by accuracy:')
top5 = results_df.sort_values('acc_mean', ascending=False).head(5)
print(top5[['model', 'feature', 'acc_mean', 'f1_mean', 'params', 'flops']].to_string(index=False))

## 9. Visualization — model × feature heatmap

In [ ]:
# Pivot to a 7-row, 3-col matrix
pivot_acc = results_df.pivot(index='model', columns='feature', values='acc_mean')
pivot_f1  = results_df.pivot(index='model', columns='feature', values='f1_mean')

# Re-order rows to the original GRID_MODELS order
pivot_acc = pivot_acc.reindex(GRID_MODELS)
pivot_f1  = pivot_f1.reindex(GRID_MODELS)
# Re-order cols
col_order = ['DWT', 'MFCC', 'LogMel']
pivot_acc = pivot_acc[col_order]
pivot_f1  = pivot_f1[col_order]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, mat, title in [(axes[0], pivot_acc, 'Accuracy'),
                       (axes[1], pivot_f1,  'Macro F1')]:
    im = ax.imshow(mat.values, cmap='viridis', vmin=0.5, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(col_order))); ax.set_xticklabels(col_order)
    ax.set_yticks(range(len(GRID_MODELS))); ax.set_yticklabels(GRID_MODELS)
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat.values[i, j]
            ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                    color='white' if v < 0.8 else 'black', fontsize=9)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'heatmap_model_feature.png'), dpi=150)
plt.show()

## 10. Visualization — params vs accuracy, FLOPs vs accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
markers = {'DWT': 'o', 'MFCC': 's', 'LogMel': '^'}
colors  = plt.cm.tab10(np.linspace(0, 1, len(GRID_MODELS)))
model_color = dict(zip(GRID_MODELS, colors))

for ax, x_col, x_label in [(axes[0], 'params', 'Trainable parameters'),
                           (axes[1], 'flops',  'FLOPs')]:
    for _, row in results_df.iterrows():
        ax.scatter(row[x_col], row['acc_mean'],
                   marker=markers[row['feature']],
                   color=model_color[row['model']],
                   s=80, edgecolor='black', linewidth=0.5)
    ax.set_xscale('log')
    ax.set_xlabel(x_label)
    ax.set_ylabel('Mean accuracy')
    ax.set_title(f'Accuracy vs {x_label}')
    ax.grid(alpha=0.3)

# Legends (split: one for models, one for features)
from matplotlib.lines import Line2D
model_handles = [Line2D([0], [0], marker='o', color='w',
                        markerfacecolor=model_color[m], markersize=8, label=m)
                 for m in GRID_MODELS]
feat_handles = [Line2D([0], [0], marker=markers[f], color='gray',
                       markersize=8, linestyle='', label=f)
                for f in col_order]
axes[0].legend(handles=model_handles, loc='lower right', fontsize=8)
axes[1].legend(handles=feat_handles,  loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'scatter_params_flops_acc.png'), dpi=150)
plt.show()

## 11. Confusion matrix for the best experiment

In [ ]:
best_row = results_df.loc[results_df['acc_mean'].idxmax()]
key = f"{best_row['model']}__{best_row['feature']}"
cm = all_cms[key]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Aggregate CM — {best_row["model"]} on {best_row["feature"]}\n'
             f'(acc = {best_row["acc_mean"]:.4f} ± {best_row["acc_std"]:.4f})')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        v = int(cm[i, j])
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_best_baseline.png'), dpi=150)
plt.show()
print(f'Best baseline: {best_row["model"]} × {best_row["feature"]} '
      f'= {best_row["acc_mean"]:.4f} acc, {best_row["params"]:,} params')

## 12. Done

Outputs in `/content/drive/MyDrive/Msc_ML_project/results_v1/`:

- `results_master.csv` — main comparison table (21 rows)
- `confusion_matrices.npz` — aggregate CMs per experiment
- `histories.json` — full training histories
- `heatmap_model_feature.png` — accuracy + F1 heatmaps
- `scatter_params_flops_acc.png` — efficiency scatter plots
- `cm_best_baseline.png` — best baseline's confusion matrix

**Next step:** Notebook C — TinyPCGNet v4.1 + Grad-CAM XAI + the final
comparison table that combines these 21 rows with v4.1's result.
